# MODFLOW 6 API A: basic use

The MODFLOW 6 application programming interface (**API**) runs MODFLOW 6 from Python one time step, or one solver iteration, at a time. Between steps, Python can read any value the model holds in memory, such as the heads, the flows, or the data of a boundary package, and change it before the model carries on. The [`modflowapi`](https://github.com/MODFLOW-ORG/modflowapi) package connects Python to MODFLOW 6.

This is the first of six notebooks on the API:

- **A. Basic use** ([`mf6-api-a-basic`](mf6-api-a-basic.ipynb)): run a model through the API, open up the solver loop, and run a flow model and a transport model together.
- **B. Watching the solver converge** ([`mf6-api-b-convergence`](mf6-api-b-convergence.ipynb)): plot the head change of every outer iteration while the model runs.
- **C. Changing recharge with a callback** ([`mf6-api-c-recharge-callback`](mf6-api-c-recharge-callback.ipynb)): change an input while the model runs.
- **D. A reverse drain** ([`mf6-api-d-reverse-drain`](mf6-api-d-reverse-drain.ipynb)): add a boundary that MODFLOW 6 does not have.
- **E. Streamflow augmentation at the coast** ([`mf6-api-e-coastal-augmentation`](mf6-api-e-coastal-augmentation.ipynb)): pump a well to keep a stream above a target flow.
- **F. Denitrification** ([`mf6-api-f-nitrate-reduction`](mf6-api-f-nitrate-reduction.ipynb)): add a chain of reactions to three transport models.

Notebooks A, B, C, and E use the coastal synthetic valley of [`synthetic-valley-a-base`](synthetic-valley-a-base.ipynb) and [`synthetic-valley-b-advanced`](synthetic-valley-b-advanced.ipynb).

## What this notebook covers

By the end of this notebook you will be able to:

- run a model with `initialize()`, `update()`, and `finalize()`,
- replace `update()` with the loop it runs for you, over the time steps and the outer iterations of each, and
- run a flow model and a transport model in one simulation, each with its own solution, with the salt the transport model simulates changing the density of the groundwater.

## How the API works

**A simulation is a set of nested parts.** A simulation holds one or more models, such as a groundwater flow (GWF) model and a groundwater transport (GWT) model. Each model holds packages, such as Recharge (RCH) or Streamflow Routing (SFR). Each model is solved by a numerical solution, which the API names `SLN_1`, `SLN_2`, and so on, in the order the simulation name file lists them.

**Time has three levels.** A stress period holds the boundary conditions constant, and it is divided into time steps. MODFLOW solves a time step by repeating **outer iterations**, each of which updates the heads, until the largest head change is smaller than the closure criterion. The solution keeps the largest head change of each outer iteration in a variable called `HNCG`, and the most outer iterations it allows in `MXITER`.

**Every API run follows the same steps.**

```text
mf6.initialize()        # read the input files and set up the model
while not finished:
    # variables changed here, between time steps
    mf6.update()        # solve one time step
mf6.finalize()          # write the rest of the output and release the memory
```

`update()` hides the outer iterations. To act inside a time step, replace it with the steps it runs:

```text
mf6.prepare_time_step(dt)
# variables changed here, once per time step
mf6.prepare_solve()
while not converged:
    # variables changed here, every outer iteration
    converged = mf6.solve()   # one outer iteration
mf6.finalize_solve()
mf6.finalize_time_step()
```

**There are two ways to read and change what the model holds.**

- A **callback** is a function that `modflowapi.run_simulation()` calls at set points of the run, such as the start of each stress period or each outer iteration. It hands the function the running simulation, whose packages are reached by name and whose stress-period data are changed like a NumPy array. Notebooks C to F use callbacks.
- **Direct memory access** reads any variable MODFLOW holds. `get_var_address()` builds the address of a variable from its name and the model and package it belongs to. `get_value()` returns a copy of the variable, which does not change when the model does. `get_value_ptr()` returns the variable itself, which stays current as the model runs and can be written to change the model.

## Imports

Import FloPy, NumPy, Matplotlib, and `ModflowApi`, the `modflowapi` class that loads the MODFLOW 6 library. `mf6_api_helpers` finds the library, and from `sv_notebook_helpers`, `BASE_MODEL` is the folder of the synthetic-valley base model and `build_variable_density()` adds a seawater transport model to a simulation.

In [ ]:
%matplotlib inline
import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import numpy as np
from mf6_api_helpers import find_mf6_libraries
from modflowapi import ModflowApi
from sv_notebook_helpers import BASE_MODEL, build_variable_density

## Find the MODFLOW 6 library

The API runs MODFLOW 6 through its shared library, `libmf6` (`libmf6.so` on Linux, `libmf6.dylib` on macOS, and `libmf6.dll` on Windows), rather than through the `mf6` program. pixi installs both with the class MODFLOW 6. `find_mf6_libraries()` returns the path of each.

In [ ]:
lib_path, mf6_exe = find_mf6_libraries()
print(f"library: {lib_path.name}")
print(f"program: {mf6_exe.name}")

## Run a model through the API

Load the coastal synthetic-valley base model with `flopy.mf6.MFSimulation.load()` and write it to a new workspace. The model has a 16,071-day steady-state stress period and 20 one-year transient periods, each of one time step, and works in meters and days.

Create a `ModflowApi` object for the library and the workspace, and call `initialize()`, which reads the input files. `get_current_time()` and `get_end_time()` give the simulation time in days. Each call to `update()` solves one time step, so the loop runs 21 times. `finalize()` writes the rest of the output.

In [ ]:
name = "sv"
ws = pl.Path("models/mf6-api-a-update")

sim = flopy.mf6.MFSimulation.load(
    sim_ws=BASE_MODEL, exe_name=mf6_exe, write_headers=False, verbosity_level=0
)
sim.set_sim_path(ws)
sim.write_simulation(silent=True)

In [ ]:
mf6 = ModflowApi(lib_path, working_directory=ws)
mf6.initialize()

current_time = mf6.get_current_time()
end_time = mf6.get_end_time()
while current_time < end_time:
    # variables changed here, between time steps
    mf6.update()
    current_time = mf6.get_current_time()
    print(f"solved to day {current_time:,.0f}")

mf6.finalize()

The library runs the same code as the `mf6` program. To check, run the same input files with the program in a second workspace and compare the heads the two runs saved.

In [ ]:
ws_program = pl.Path("models/mf6-api-a-program")
sim.set_sim_path(ws_program)
sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"

head_api = flopy.utils.HeadFile(ws / "sv.hds").get_alldata()
head_program = flopy.utils.HeadFile(ws_program / "sv.hds").get_alldata()
active = head_program < 1.0e29
difference = np.abs(head_api - head_program)[active].max()
print(f"largest head difference in any cell and time step: {difference:.1e} m")

**What to look for.** The two runs give the same head in every cell at every time step, to the last digit the head file stores, because the library and the program run the same code on the same input files.

## Open up the solver loop

Replace `update()` with the steps it runs. Read `MXITER`, the most outer iterations the solution allows, with `get_value()`, which returns a copy as a NumPy array. Read the stress period from `KPER` in the Time Discretization package (TDIS), counted from 1. Within each time step, call `solve()` until it returns `True`, which means the outer iterations have converged, or until `MXITER` outer iterations have run.

The comment `# variables changed here` marks where Python can change a variable the model holds. Between two calls to `solve()`, it can change an input that depends on the current heads, as notebooks D and E do.

In [ ]:
ws = pl.Path("models/mf6-api-a-solver-loop")
sim = flopy.mf6.MFSimulation.load(
    sim_ws=BASE_MODEL, exe_name=mf6_exe, write_headers=False, verbosity_level=0
)
sim.set_sim_path(ws)
sim.write_simulation(silent=True)

mf6 = ModflowApi(lib_path, working_directory=ws)
mf6.initialize()
max_iter = int(mf6.get_value(mf6.get_var_address("MXITER", "SLN_1"))[0])
kper_address = mf6.get_var_address("KPER", "TDIS")

iterations = {}
while mf6.get_current_time() < mf6.get_end_time():
    mf6.prepare_time_step(mf6.get_time_step())
    kper = int(mf6.get_value(kper_address)[0])

    # variables changed here, once per time step
    mf6.prepare_solve()
    kiter = 0
    converged = False
    while kiter < max_iter and not converged:
        # variables changed here, every outer iteration
        converged = mf6.solve()
        kiter += 1
    if not converged:
        print(f"stress period {kper} did not converge")
    mf6.finalize_solve()

    mf6.finalize_time_step()
    iterations[kper] = kiter

mf6.finalize()
print("outer iterations in each stress period:")
print(iterations)

**What to look for.** The steady-state period converges in 6 outer iterations, because the model starts from its own steady-state heads. Each transient period starts from the heads of the year before and takes 2 to 15 outer iterations. The most, 15, are in stress period 20, the year 2024. Notebook B plots the head change of each of these iterations.

## Run a flow model and a transport model together

A simulation can hold several models, each with its own solution. Add a groundwater transport (GWT) model to the base model, to simulate the salt that seawater carries into the aquifer along the coast. The General-Head Boundary (GHB) along the coast already lists the salt concentration of seawater, 35 kilograms per cubic meter (kg/m³), and the elevation of each boundary as auxiliary variables named `concentration` and `elevation`. The Source and Sink Mixing (**SSM**) package gives that concentration to the water the GHB brings into the aquifer. Water that leaves the aquifer through any boundary carries the concentration of its cell, and recharge carries no salt.

Seawater is denser than fresh groundwater. The density of water rises by 0.7 kg/m³ for each kg/m³ of salt, so seawater is 1,024.5 kg/m³. Below sea level, the sea presses on the aquifer harder than fresh water at sea level would, and salty groundwater sinks beneath fresh groundwater. The Buoyancy (**BUY**) package adds both to the flow model. It works out the density of the water in each cell from the salt concentration of the transport model, and the density of the sea at each GHB from its `concentration` auxiliary variable, at the depth of its `elevation` auxiliary variable. The coastal heads stay at sea level, 0 m. The flow solution is solved before the transport solution in each time step, so BUY uses the concentrations of the time step before.

The transport model has the grid of the flow model and starts with no salt. The transport time steps are those of the flow model, so the first is the 16,071-day steady-state period. Its packages are the Discretization (DIS), Initial Conditions (IC), Mobile Storage and Transfer (MST, which sets the porosity), Advection (ADV), Dispersion (DSP), SSM, and Output Control (OC) packages. The GWF-GWT exchange passes the flows of each time step from the flow model to the transport model. Each model needs its own Iterative Model Solution (IMS), and `register_solution_package()` assigns the new IMS to the transport model.

Load the base model and pass it, with a new workspace, to `build_variable_density()` in `sv_notebook_helpers`. It writes a copy of the simulation to the workspace, adds BUY, the transport model, its IMS, and the exchange to the copy, and returns the copy, ready to run. The simulation passed in is not changed. Get the flow model (`sv`) and the transport model (`salt`) from it to map the results below.

In [ ]:
ws = pl.Path("models/mf6-api-a-flow-transport")
base = flopy.mf6.MFSimulation.load(
    sim_ws=BASE_MODEL, exe_name=mf6_exe, write_headers=False, verbosity_level=0
)
sim = build_variable_density(base, ws)
gwf = sim.get_model(name)
gwt = sim.get_model("salt")
dis = gwf.dis

Run the two models through the API. In every time step, solve the flow solution (`component_id=1`) to convergence and then the transport solution (`component_id=2`), which uses the flows just solved. The loop is the one above, run once for each solution.

In [ ]:
mf6 = ModflowApi(lib_path, working_directory=ws)
mf6.initialize()
max_iter = {
    km: int(mf6.get_value(mf6.get_var_address("MXITER", f"SLN_{km}"))[0])
    for km in (1, 2)
}

iterations = {1: [], 2: []}
while mf6.get_current_time() < mf6.get_end_time():
    mf6.prepare_time_step(mf6.get_time_step())
    for km in (1, 2):  # flow, then transport
        mf6.prepare_solve(km)
        kiter = 0
        converged = False
        while kiter < max_iter[km] and not converged:
            converged = mf6.solve(km)
            kiter += 1
        if not converged:
            print(f"solution {km} did not converge")
        mf6.finalize_solve(km)
        iterations[km].append(kiter)
    mf6.finalize_time_step()

mf6.finalize()
print(f"flow: {iterations[1]}")
print(f"transport: {iterations[2]}")

Map the salt at the end of the simulation in layer 1, the top of the upper aquifer, and in layer 5, the bottom of the lower aquifer, with `flopy.plot.PlotMapView()`. The salt is shown as the percentage of seawater in the groundwater, its concentration divided by that of seawater, 35 kg/m³. Groundwater with less than 0.1 percent seawater is left blank. Mark the lake, the river, and the wells on both layers so the salt can be placed in the valley. The lake is the layer 1 cells with a hydraulic conductivity above 1,000 m/d, and the river is the cells of the River (RIV) package. The two supply wells (`pwell`) and the prediction well (`prediction`) are screened in layers 4 and 5.

In [ ]:
width = 17.15 / 2.54  # figure width, inches
# the salt as a percentage of seawater, 35 kg/m³
conc = gwt.output.concentration().get_data()
percent = 100.0 * conc / 35.0
percent = np.ma.masked_where((conc >= 1.0e29) | (percent < 0.1), percent)
panels = {"A": (0, "Layer 1, upper aquifer"), "B": (4, "Layer 5, lower aquifer")}

# the lake and river cells, and the row and column of each well
lake = gwf.npf.k.array[0] > 1000.0
lake_mask = np.ma.masked_where(~lake, np.ones(lake.shape))
river = np.zeros(lake.shape, dtype=bool)
for _, i, j in gwf.get_package("riv-1").stress_period_data.get_data(0)["cellid"]:
    river[i, j] = True
river_mask = np.ma.masked_where(~river, np.ones(river.shape))
xc, yc = gwf.modelgrid.xcellcenters, gwf.modelgrid.ycellcenters
wells = {}
for package, kper in (("pwell", 1), ("prediction", 11)):
    cells = gwf.get_package(package).stress_period_data.get_data(kper)["cellid"]
    rows_cols = sorted({(i, j) for _, i, j in cells})
    wells[package] = (
        [xc[i, j] for i, j in rows_cols],
        [yc[i, j] for i, j in rows_cols],
    )

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        [list(panels)], figsize=(width, 0.7 * width), layout="constrained"
    )
    for letter, (k, heading) in panels.items():
        ax = axd[letter]
        mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=k)
        mm.plot_inactive(color_noflow="0.85")
        pc = mm.plot_array(
            percent[k], cmap="viridis", norm=plt.matplotlib.colors.LogNorm(0.1, 100.0)
        )
        mm.plot_grid(lw=0.2, color="0.7")
        mm.plot_array(lake_mask, cmap="Blues", vmin=0.0, vmax=1.5)
        mm.plot_array(river_mask, cmap=plt.matplotlib.colors.ListedColormap(["cyan"]))
        x, y = wells["pwell"]
        ax.plot(x, y, ls="none", marker="o", ms=5, mfc="white", mec="black")
        x, y = wells["prediction"]
        ax.plot(x, y, ls="none", marker="*", ms=9, mfc="#eb6834", mec="black")
        ax.set_aspect("equal")
        flopy.plot.styles.heading(ax=ax, letter=letter, heading=heading)
        flopy.plot.styles.xlabel(ax=ax, label="x (m)")
        flopy.plot.styles.ylabel(ax=ax, label="y (m)")
    cbar = fig.colorbar(
        pc, ax=list(axd.values()), shrink=0.6, label="Seawater (percent)"
    )
    cbar.set_ticks([0.1, 1.0, 10.0, 100.0], labels=["0.1", "1", "10", "100"])
    Patch, Line2D = plt.matplotlib.patches.Patch, plt.matplotlib.lines.Line2D
    handles = [
        Patch(color=plt.get_cmap("Blues")(1.0 / 1.5), label="Lake"),
        Patch(color="cyan", label="River"),
        Line2D(
            [],
            [],
            ls="none",
            marker="o",
            ms=5,
            mfc="white",
            mec="black",
            label="Supply well",
        ),
        Line2D(
            [],
            [],
            ls="none",
            marker="*",
            ms=9,
            mfc="#eb6834",
            mec="black",
            label="Prediction well",
        ),
    ]
    fig.legend(handles=handles, loc="outside lower center", ncols=4, frameon=False)

# how far inland the salt reaches, from the cell center of the coastal row
nrow = dis.nrow.data
delc = dis.delc.array[0]
for k, label in ((0, "layer 1"), (4, "layer 5")):
    line = f"{label}: up to {percent[k].max():.1f} percent seawater"
    for threshold in (1.0, 50.0):
        rows = np.nonzero((percent[k] >= threshold).filled(False).any(axis=1))[0]
        reach = (nrow - 1 - rows.min()) * delc if rows.size else 0.0
        line += f"; {threshold:.0f} percent reaches {reach:,.0f} m inland"
    print(line)

**What to look for.** BUY changes the flow solution, so the flow model takes a different number of outer iterations than in the run above, most (9) in the steady-state period and in stress periods 14 and 18. The transport model takes 7 outer iterations in the steady-state period and 2 or 3 in each year after it. By the end of the simulation, seawater has come into the aquifer along most of the coast. Salty water sinks, so the salt collects at the bottom of the lower aquifer. The groundwater is up to 98.6 percent seawater in layer 5 but only 63.9 percent in layer 1. Water that is half seawater reaches 914 m inland in layer 5, but only 152 m, one row, in layer 1, at the river. Water that is 1 percent seawater reaches 1,219 m inland in both layers: at the river in layer 1, and between the `vc` supply well (the circle near the coast), which pumps from layers 4 and 5, and the river in layer 5. The distances are from the center of the coastal row. The prediction well of notebook E (star) is near the inland edge of the salt in layer 5.

### Try it: one fluid density

Notebook E uses a simpler model of the coast, with one fluid density. It gives the coastal boundaries the weight of the sea by raising their heads to **equivalent freshwater heads**, the head of a column of fresh water that presses as hard at the depth of each boundary as the sea does. To try it here, pass `buoyancy=False` to `build_variable_density()` in the cell that builds the transport model. In place of BUY, it raises the coastal GHB heads with

```python
ghb = gwf.get_package("ghb-1")
data = ghb.stress_period_data.get_data(0)
ratio = (1000.0 + 0.7 * data["concentration"]) / 1000.0
data["bhead"] += (ratio - 1.0) * (data["bhead"] - data["elevation"])
ghb.stress_period_data.set_data({0: data})
```

Run the notebook again from that cell. The sea still pushes salt into the aquifer, but the salty water no longer sinks. Where does the salt go in each layer, and how far inland does it reach? (With one density, the groundwater is up to 96.8 percent seawater in layer 1, water that is half seawater reaches 305 m inland in layer 1 and 610 m in layer 5, water that is 1 percent seawater reaches 1,067 m inland in both layers, and the run takes about as long as with BUY.)

## Recap

You ran the synthetic-valley base model through the MODFLOW 6 library with `initialize()`, `update()`, and `finalize()`, and checked that it gives the same heads as the `mf6` program. You then replaced `update()` with the time-step and outer-iteration loop it runs, and ran a flow model and a transport model of seawater salt in one simulation, solving each with its own solution in every time step. The Buoyancy package passed the density of the salty water back to the flow model, so the salt sank to the bottom of the lower aquifer. The other notebooks in the series act inside these loops.